In [41]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,FunctionTransformer
from sklearn.metrics import confusion_matrix,f1_score,accuracy_score,precision_score,recall_score
import joblib

In [4]:
rfm = pd.read_csv("../data/clients_rfm.csv")

In [5]:
rfm["Cluster"].value_counts()

Cluster
Bronze    1923
Gold      1689
Silver     726
Name: count, dtype: int64

In [ ]:
features = ["Recency","Frequency","Monetary"]
X_train, X_test, y_train, y_test = train_test_split(rfm[features],rfm["Cluster"],test_size=0.2,random_state=42)

3975    Bronze
1448    Bronze
2662      Gold
17        Gold
1634    Bronze
         ...  
3466      Gold
2643      Gold
416     Silver
1616    Bronze
2478    Bronze
Name: Cluster, Length: 868, dtype: str

In [ ]:

params = {
    "n_estimators":[200,500,1000],
    "max_depth":[None,5,10,20,30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5, 1.0],
    "class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    RandomForestClassifier(),
    param_distributions = params,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)


search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)


{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30, 'class_weight': None}
0.9812680115273775


selon les recherches effectuées par randomizedsearchcv on trouve que {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30, 'class_weight': None} sont les meilleurs valeurs identifié en se basant sur le metric accuracy

In [39]:
model =RandomForestClassifier(
    n_estimators=500,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    max_depth=30,
    class_weight=None
)
model.fit(X=X_train,y=y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred,average="weighted")
recall = recall_score(y_test, y_pred,average="weighted")
f1 = f1_score(y_test, y_pred,average="weighted")
cm = confusion_matrix(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
print("\nConfusion Matrix:")
print(cm)

importance = pd.DataFrame({
    'Variable': ['Recence', 'Frequence', 'Montant'],
    'Importance': model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)


Accuracy : 0.9838709677419355
Precision: 0.9839260425200059
Recall   : 0.9838709677419355
F1-score : 0.983867321467679

Confusion Matrix:
[[385   3   0]
 [  4 322   5]
 [  0   2 147]]
    Variable  Importance
1  Frequence    0.353273
2    Montant    0.344545
0    Recence    0.302182


In [42]:
joblib.dump(model,"../artifacts/classification_model.pkl")

['../artifacts/classification_model.pkl']